# ML1. Введение в машинное обучение

Русифицированная версия решения проекта School 21.  
Датасет: Kaggle `Two Sigma Connect: Rental Listing Inquiries`, табличный файл `train.json`.

Цель проекта: провести первичный анализ данных об объявлениях аренды квартир, подготовить простые признаки и обучить несколько базовых регрессионных моделей для предсказания цены аренды.

## 1. Введение

### 1.1 Пять примеров применения ML в жизни

1. **Фильтрация спама в почте.** Модель автоматически отделяет полезные письма от нежелательных и экономит время пользователя.
2. **Кредитный скоринг в банке.** ML оценивает вероятность возврата кредита и помогает банку управлять финансовыми рисками.
3. **Анализ медицинских снимков.** ML может находить подозрительные области на снимках и помогать врачу быстрее поставить диагноз.
4. **Рекомендательные системы.** ML подбирает фильмы, музыку, товары или курсы под интересы пользователя.
5. **Прогнозирование спроса в магазинах.** ML помогает заранее оценить будущие продажи и правильно планировать запасы.

### 1.2 Классы задач

Примеры из таблицы теории можно отнести к таким классам:

| Случай | Возможный класс ML-задачи |
| --- | --- |
| Предсказать цену дома | Регрессия |
| Предсказать, вернёт ли клиент кредит | Бинарная классификация; иногда регрессия, если предсказываем ожидаемый убыток |
| Предсказать, когда пациенту нужно принять лекарство | Классификация или регрессия/прогноз времени |
| Выбрать лекарство для пациента | Многоклассовая классификация или рекомендательная задача |
| Выбрать сегмент клиентов для акции | Классификация, регрессия, кластеризация, uplift-моделирование |
| Распознать дефект по фото | Бинарная классификация |
| Решить, как расставить товары на полке | Регрессия/оптимизация, иногда reinforcement learning |
| Поиск сайтов по текстовому запросу | Ранжирование, information retrieval |
| Разделить покупателей на сегменты | Кластеризация |
| Обнаружить аномалию в трафике сайта | Поиск аномалий, unsupervised/semi-supervised learning |

Мои пять примеров:

| Пример | Класс задачи |
| --- | --- |
| Фильтрация спама | Бинарная классификация |
| Кредитный скоринг | Бинарная классификация или регрессия |
| Анализ медицинских снимков | Классификация или сегментация |
| Рекомендательные системы | Ранжирование, классификация, ассоциативные правила |
| Прогноз спроса | Регрессия, прогнозирование временных рядов |

### 1.3 Multiclass и multilabel

В **многоклассовой классификации** объект относится ровно к одному классу из нескольких возможных. Например, фрукт может быть яблоком, грушей или бананом.

В **многометочной классификации** объект может иметь несколько меток одновременно. Например, фильм может быть и комедией, и мелодрамой.

### 1.4 Цены на дома: классификация или регрессия?

Пример с ценами на дома — это **регрессия**, потому что целевая переменная является непрерывным числом. Регрессию можно свести к классификации, если разбить цены на группы, например `дешёвые`, `средние`, `дорогие`. Но при таком подходе теряется часть информации: точные цены заменяются грубыми категориями.

## 2. Введение в анализ данных

In [ ]:
# Импортируем библиотеки для анализа данных, визуализации и обучения моделей.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("lightgbm is not installed in this environment. Install it if your Jupyter environment requires a direct import.")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 100)

In [ ]:
# Ищем train.json в нескольких стандартных местах, чтобы ноутбук запускался из корня проекта и из папки src.
def find_train_json():
    candidates = [
        Path("src/train.json"),
        Path("datasets/train.json"),
        Path("data/train.json"),
        Path("train.json"),
        Path("../src/train.json"),
        Path("../datasets/train.json"),
        Path("../data/train.json"),
        Path("../train.json"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "train.json was not found. Download it from Kaggle and place it into datasets/train.json "
        "or data/train.json."
    )


DATA_PATH = find_train_json()
data = pd.read_json(DATA_PATH)
data.head()

In [ ]:
# Проверяем размер таблицы: количество строк и столбцов.
print(f"Размер датасета: {data.shape[0]} строк, {data.shape[1]} столбцов")

In [ ]:
# Смотрим список столбцов и явно фиксируем целевую переменную.
print("Столбцы:")
print(data.columns.tolist())
print("\nЦелевая переменная: price")

In [ ]:
# Изучаем типы данных и количество непустых значений.
data.info()

In [ ]:
# Считаем описательную статистику только для числовых столбцов.
data.describe().T

In [ ]:
# Для нечисловых столбцов делаем лёгкий обзор без тяжёлой обработки списков.
non_numeric_overview = pd.DataFrame(
    {
        "dtype": data.select_dtypes(exclude=np.number).dtypes.astype(str),
        "non_null": data.select_dtypes(exclude=np.number).notna().sum(),
        "sample_value": data.select_dtypes(exclude=np.number).iloc[0].astype(str),
    }
)
non_numeric_overview

In [ ]:
# Считаем корреляции между числовыми признаками.
numeric_corr = data.select_dtypes(include=np.number).corr(numeric_only=True)
numeric_corr

In [ ]:
# Проверяем пропуски и долю пропущенных значений по каждому столбцу.
missing_summary = (
    data.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)
missing_summary["missing_percent"] = missing_summary["missing_count"] / len(data) * 100
missing_summary.head(20)

### 2.5 Краткие выводы по первичному анализу

Для исходного Kaggle-файла `train.json` ожидаемый размер данных: **49352 строки и 15 столбцов**.

`info()` показывает типы столбцов и количество непустых значений. С его помощью удобно быстро увидеть числовые, строковые и объектные признаки.

`describe()` показывает базовую описательную статистику. Для числовых столбцов это среднее, стандартное отклонение, минимум, максимум и квартили.

`corr()` показывает линейные корреляции между числовыми столбцами. Значение около `1` означает сильную положительную связь, около `-1` — сильную отрицательную, около `0` — слабую линейную связь.

Таблица пропусков показывает, есть ли в данных `NaN`. В исходной таблице нет полностью пустых столбцов. При этом в текстовых и списковых столбцах могут встречаться пустые строки или пустые списки, поэтому одного `isna()` не всегда достаточно для глубокого анализа качества данных.

In [ ]:
# Оставляем только признаки и target, которые нужны по заданию.
work_data = data[["bathrooms", "bedrooms", "interest_level", "price"]].copy()
work_data.head()

## 3. Статистический анализ данных

### 3.3 Анализ целевой переменной `price`

In [ ]:
# Строим гистограмму целевой переменной до удаления выбросов.
plt.figure(figsize=(10, 5))
sns.histplot(work_data["price"], bins=80, kde=True)
plt.title("Распределение price до фильтрации выбросов")
plt.xlabel("Price")
plt.ylabel("Количество")
plt.show()

Первая гистограмма обычно имеет длинный правый хвост: большая часть цен находится в относительно небольшом диапазоне, а редкие очень дорогие объявления сильно растягивают шкалу.

In [ ]:
# Boxplot помогает визуально увидеть выбросы в цене.
plt.figure(figsize=(10, 3))
sns.boxplot(x=work_data["price"])
plt.title("Boxplot price до фильтрации выбросов")
plt.xlabel("Price")
plt.show()

Boxplot хорошо показывает выбросы. Цены далеко выше верхнего уса могут быть объявлениями премиум-класса, ошибками данных или просто редкими случаями. Такие значения могут сильно влиять на простые модели и графики.

In [ ]:
# Удаляем строки ниже 1-го и выше 99-го перцентиля цены.
lower_price = work_data["price"].quantile(0.01)
upper_price = work_data["price"].quantile(0.99)

filtered_data = work_data[
    work_data["price"].between(lower_price, upper_price)
].copy()

print(f"1-й перцентиль: {lower_price:.2f}")
print(f"99-й перцентиль: {upper_price:.2f}")
print(f"Строк до фильтрации: {len(work_data)}")
print(f"Строк после фильтрации: {len(filtered_data)}")

In [ ]:
# Повторно строим гистограмму цены после фильтрации выбросов.
plt.figure(figsize=(10, 5))
sns.histplot(filtered_data["price"], bins=60, kde=True)
plt.title("Распределение price после фильтрации 1%-99%")
plt.xlabel("Price")
plt.ylabel("Количество")
plt.show()

После удаления нижнего 1% и верхнего 1% цен распределение становится заметно понятнее. Целевая переменная всё ещё скошена вправо, но экстремальные значения уже не доминируют на графике.

### 3.4 Анализ признаков

In [ ]:
# Проверяем тип категориального признака interest_level.
print(f"Тип interest_level: {filtered_data['interest_level'].dtype}")

In [ ]:
# Считаем, сколько раз встречается каждый уровень интереса.
filtered_data["interest_level"].value_counts()

In [ ]:
# Кодируем категории числами: low, medium и high превращаются в 0, 1 и 2.
interest_level_map = {"low": 0, "medium": 1, "high": 2}
filtered_data["interest_level_encoded"] = filtered_data["interest_level"].map(interest_level_map)
filtered_data.head()

In [ ]:
# Строим гистограммы для количества ванных комнат и спален.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(filtered_data["bathrooms"], bins=20, ax=axes[0])
axes[0].set_title("Распределение bathrooms")
axes[0].set_xlabel("Bathrooms")

sns.histplot(filtered_data["bedrooms"], bins=20, ax=axes[1])
axes[1].set_title("Распределение bedrooms")
axes[1].set_xlabel("Bedrooms")

plt.tight_layout()
plt.show()

In [ ]:
# Проверяем bathrooms и bedrooms через boxplot.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=filtered_data["bathrooms"], ax=axes[0])
axes[0].set_title("Boxplot bathrooms")
axes[0].set_xlabel("Bathrooms")

sns.boxplot(x=filtered_data["bedrooms"], ax=axes[1])
axes[1].set_title("Boxplot bedrooms")
axes[1].set_xlabel("Bedrooms")

plt.tight_layout()
plt.show()

`interest_level` — категориальный строковый признак. Мы кодируем его как `low = 0`, `medium = 1`, `high = 2`, чтобы использовать его в корреляционном анализе и простых числовых моделях.

После фильтрации `price` по 1-му и 99-му перцентилям ожидаемые количества значений: `low = 33697`, `medium = 11116`, `high = 3566`.

Гистограммы и boxplot для `bathrooms` и `bedrooms` показывают, что большинство объявлений имеет небольшое количество ванных комнат и спален. Согласно чеклисту проекта, после основной фильтрации целевой переменной критичных выбросов для этих двух признаков нет.

### 3.5 Комплексный анализ

In [ ]:
# Формируем корреляционную матрицу для базовых признаков и цены.
corr_columns = ["bathrooms", "bedrooms", "interest_level_encoded", "price"]
corr_matrix = filtered_data[corr_columns].corr()
corr_matrix

In [ ]:
# Визуализируем корреляции с помощью heatmap.
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Корреляционная матрица")
plt.show()

Heatmap показывает линейные связи между признаками и ценой. В этом маленьком наборе признаков корреляции есть, но они не очень сильные. Среди базовых признаков `bathrooms` обычно сильнее всего коррелирует с `price`, однако только `bathrooms` и `bedrooms` недостаточно, чтобы хорошо объяснить цену аренды.

In [ ]:
# Строим scatterplot: цена по оси X, каждый признак по оси Y.
features_for_scatter = ["bathrooms", "bedrooms", "interest_level_encoded"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature in zip(axes, features_for_scatter):
    sns.scatterplot(data=filtered_data, x="price", y=feature, alpha=0.35, ax=ax)
    ax.set_title(f"Price vs {feature}")
    ax.set_xlabel("Price")
    ax.set_ylabel(feature)

plt.tight_layout()
plt.show()

## 4. Создание признаков

In [ ]:
# Создаём новые квадратичные признаки и смотрим, меняется ли связь с ценой.
feature_data = filtered_data.copy()
feature_data["bathrooms_squared"] = feature_data["bathrooms"] ** 2
feature_data["bedrooms_squared"] = feature_data["bedrooms"] ** 2
feature_data["interest_level_squared"] = feature_data["interest_level_encoded"] ** 2

created_feature_columns = [
    "bathrooms",
    "bedrooms",
    "interest_level_encoded",
    "bathrooms_squared",
    "bedrooms_squared",
    "interest_level_squared",
    "price",
]

created_corr = feature_data[created_feature_columns].corr()
created_corr

In [ ]:
# Визуализируем корреляции после добавления квадратичных признаков.
plt.figure(figsize=(9, 6))
sns.heatmap(created_corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Корреляционная матрица с квадратичными признаками")
plt.show()

In [ ]:
# Сортируем корреляции признаков с target по модулю.
target_corr = created_corr["price"].drop("price").sort_values(key=lambda s: s.abs(), ascending=False)
target_corr

Квадратичные признаки могут изменить корреляцию с ценой, потому что они сильнее подчёркивают большие значения. В ожидаемом результате проекта квадратичные признаки не дают более высокой корреляции с `price`, чем исходные базовые признаки. Важно помнить: корреляция сама по себе не доказывает, что признак улучшит качество модели.

In [ ]:
# Для обучения берём только bathrooms и bedrooms, затем делим данные на train и test.
X = filtered_data[["bathrooms", "bedrooms"]].copy()
y = filtered_data["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=21,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Создаём полиномиальные признаки степени 10 для линейной регрессии.
poly = PolynomialFeatures(degree=10, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f"Polynomial train shape: {X_train_poly.shape}")
print(f"Polynomial test shape: {X_test_poly.shape}")

## 5. Модели

In [ ]:
# Подготавливаем таблицы метрик и вспомогательные функции для MAE/RMSE.
result_MAE = pd.DataFrame(columns=["model", "train", "test"])
result_RMSE = pd.DataFrame(columns=["model", "train", "test"])


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def add_metrics(model_name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    global result_MAE, result_RMSE
    result_MAE.loc[len(result_MAE)] = [
        model_name,
        mean_absolute_error(y_train_true, y_train_pred),
        mean_absolute_error(y_test_true, y_test_pred),
    ]
    result_RMSE.loc[len(result_RMSE)] = [
        model_name,
        rmse(y_train_true, y_train_pred),
        rmse(y_test_true, y_test_pred),
    ]

### 5.2 Линейная регрессия

In [ ]:
# Обучаем линейную регрессию на полиномиальных признаках и сохраняем предсказания.
linear_regression = LinearRegression()
linear_regression.fit(X_train_poly, y_train)

train_linear_pred = linear_regression.predict(X_train_poly)
test_linear_pred = linear_regression.predict(X_test_poly)

train_predictions = X_train.copy()
test_predictions = X_test.copy()
train_predictions["target"] = y_train
test_predictions["target"] = y_test

train_predictions["linear_regression_prediction"] = train_linear_pred
test_predictions["linear_regression_prediction"] = test_linear_pred

add_metrics(
    "linear_regression",
    y_train,
    train_linear_pred,
    y_test,
    test_linear_pred,
)

result_MAE, result_RMSE

### 5.3 Дерево решений

In [ ]:
# Обучаем дерево решений на исходных признаках bathrooms и bedrooms.
decision_tree = DecisionTreeRegressor(random_state=21)
decision_tree.fit(X_train, y_train)

train_tree_pred = decision_tree.predict(X_train)
test_tree_pred = decision_tree.predict(X_test)

train_predictions["decision_tree_prediction"] = train_tree_pred
test_predictions["decision_tree_prediction"] = test_tree_pred

add_metrics(
    "decision_tree",
    y_train,
    train_tree_pred,
    y_test,
    test_tree_pred,
)

result_MAE, result_RMSE

### 5.4 Наивные модели

In [ ]:
# Строим наивные baseline-модели: предсказание средним и медианой цены train-выборки.
train_mean_price = y_train.mean()
train_median_price = y_train.median()

train_mean_pred = np.full(shape=len(y_train), fill_value=train_mean_price)
test_mean_pred = np.full(shape=len(y_test), fill_value=train_mean_price)

train_median_pred = np.full(shape=len(y_train), fill_value=train_median_price)
test_median_pred = np.full(shape=len(y_test), fill_value=train_median_price)

train_predictions["naive_mean_prediction"] = train_mean_pred
test_predictions["naive_mean_prediction"] = test_mean_pred
train_predictions["naive_median_prediction"] = train_median_pred
test_predictions["naive_median_prediction"] = test_median_pred

add_metrics("naive_mean", y_train, train_mean_pred, y_test, test_mean_pred)
add_metrics("naive_median", y_train, train_median_pred, y_test, test_median_pred)

result_MAE, result_RMSE

### 5.5 Сравнение результатов

In [ ]:
# Показываем итоговую таблицу MAE.
result_MAE

In [ ]:
# Показываем итоговую таблицу RMSE.
result_RMSE

In [ ]:
# Выбираем лучшую модель по минимальной ошибке на тестовой выборке.
best_mae_model = result_MAE.sort_values("test").iloc[0]
best_rmse_model = result_RMSE.sort_values("test").iloc[0]

print(f"Лучшая модель по test MAE: {best_mae_model['model']} ({best_mae_model['test']:.2f})")
print(f"Лучшая модель по test RMSE: {best_rmse_model['model']} ({best_rmse_model['test']:.2f})")

Лучшая модель — та, у которой минимальная ошибка на тестовой выборке. Тестовые метрики важнее тренировочных, потому что они показывают, как модель работает на новых данных.

В этом проекте мы используем только `bathrooms` и `bedrooms`, поэтому качество моделей ограничено. Реальная цена аренды также зависит от района, текста объявления, фотографий, характеристик здания и многих других факторов.

## Дополнительная практика

In [ ]:
# Дополнительно создаём несколько признаков из текста, списков и даты.
additional_data = data.copy()

additional_data["description_length"] = additional_data["description"].fillna("").str.len()
additional_data["features_count"] = additional_data["features"].apply(lambda x: len(x) if isinstance(x, list) else 0)
additional_data["photos_count"] = additional_data["photos"].apply(lambda x: len(x) if isinstance(x, list) else 0)
additional_data["created"] = pd.to_datetime(additional_data["created"], errors="coerce")
additional_data["created_month"] = additional_data["created"].dt.month
additional_data["created_dayofweek"] = additional_data["created"].dt.dayofweek

additional_columns = [
    "bathrooms",
    "bedrooms",
    "latitude",
    "longitude",
    "description_length",
    "features_count",
    "photos_count",
    "created_month",
    "created_dayofweek",
    "price",
]

additional_data[additional_columns].head()